# LanceDB Track Catalog Indexing

This notebook is an operator wrapper around the checked-in LanceDB files. It does not duplicate indexing logic in notebook cells.

Files called by this notebook:

- `scripts/build_lancedb_index.py` builds the local LanceDB table.
- `mcrs/lancedb/indexing.py` owns row construction, FTS index creation, and embedding-column handling.
- `modal/app.py::upload_lancedb_index` uploads the built DB into the `music-crs-models` Modal volume; its `--overwrite` option replaces an existing `/lancedb` target.
- `scripts/smoke_lancedb_modal_query.py` smoke-tests the deployed private Modal query function.

By default, the build stores precomputed track embedding columns. Set `INCLUDE_EMBEDDINGS = False` to add `--metadata-only` for a smaller sparse-only artifact.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "mcrs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "scripts" / "build_lancedb_index.py").exists(), PROJECT_ROOT
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
OUT_DIR = PROJECT_ROOT / "cache" / "lancedb"
TABLE_NAME = "music_track_catalog"
DROP_EXISTING = True
BATCH_SIZE = 1024

# True is the script default. False adds --metadata-only.
INCLUDE_EMBEDDINGS = True

# Optional Modal steps stay off until you intentionally flip them on.
UPLOAD_TO_MODAL = False
OVERWRITE_MODAL_INDEX = True
REMOTE_DIR = "lancedb"
RUN_MODAL_SMOKE = False
SMOKE_QUERY = "dark atmospheric synthwave"
SMOKE_TOPK = 20

print("Local DB:", OUT_DIR)
print("Embedding columns:", "included" if INCLUDE_EMBEDDINGS else "metadata-only")

In [ ]:
def run_command(args):
    args = [str(arg) for arg in args]
    print("$ " + " ".join(shlex.quote(arg) for arg in args))
    result = subprocess.run(
        args,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
    return result

## Build the local LanceDB index

This cell calls `scripts/build_lancedb_index.py`. The notebook intentionally relies on the script default for embedding columns; it only passes `--metadata-only` when `INCLUDE_EMBEDDINGS` is set to `False`.

In [ ]:
build_cmd = [
    "uv",
    "run",
    "python",
    "scripts/build_lancedb_index.py",
    "--out-dir",
    OUT_DIR,
    "--table-name",
    TABLE_NAME,
    "--batch-size",
    BATCH_SIZE,
]
if DROP_EXISTING:
    build_cmd.append("--drop-existing")
if not INCLUDE_EMBEDDINGS:
    build_cmd.append("--metadata-only")

build_result = run_command(build_cmd)
build_summary = json.loads(build_result.stdout)
display(pd.DataFrame([build_summary]).T.rename(columns={0: "value"}))

## Verify table schema

This checks the local DB produced by the external script and confirms whether the expected embedding vector fields and `has_*` flags are present.

In [ ]:
import lancedb

from mcrs.milvus.indexing import EMBEDDING_FIELDS, has_embedding_field_name, milvus_safe_field_name

db = lancedb.connect(str(OUT_DIR))
table = db.open_table(TABLE_NAME)
schema = table.schema
if callable(schema):
    schema = schema()
schema_names = set(schema.names)

embedding_schema_rows = []
for raw_name in EMBEDDING_FIELDS:
    vector_field = milvus_safe_field_name(raw_name)
    has_field = has_embedding_field_name(vector_field)
    embedding_schema_rows.append(
        {
            "raw_embedding": raw_name,
            "vector_field": vector_field,
            "vector_present": vector_field in schema_names,
            "has_flag": has_field in schema_names,
        }
    )

display(pd.DataFrame(embedding_schema_rows))
print(f"Rows in {TABLE_NAME}: {table.count_rows():,}")

## Optional: upload to Modal models volume

Flip `UPLOAD_TO_MODAL = True` in the configuration cell to upload the local DB directory to `/root/models/lancedb`.

In [ ]:
if UPLOAD_TO_MODAL:
    upload_cmd = [
        "uv",
        "run",
        "modal",
        "run",
        "modal/app.py::upload_lancedb_index",
        "--local-db-dir",
        OUT_DIR,
        "--remote-dir",
        REMOTE_DIR,
    ]
    if OVERWRITE_MODAL_INDEX:
        upload_cmd.append("--overwrite")
    run_command(upload_cmd)
else:
    print("Skipping Modal upload. Set UPLOAD_TO_MODAL = True to run it.")

## Optional: smoke-test the deployed Modal query function

Deploy `modal/app.py` first if needed, then flip `RUN_MODAL_SMOKE = True`.

In [ ]:
if RUN_MODAL_SMOKE:
    smoke_cmd = [
        "uv",
        "run",
        "python",
        "scripts/smoke_lancedb_modal_query.py",
        "--query",
        SMOKE_QUERY,
        "--topk",
        SMOKE_TOPK,
        "--remote-db-uri",
        f"/root/models/{REMOTE_DIR}",
    ]
    run_command(smoke_cmd)
else:
    print("Skipping Modal smoke. Set RUN_MODAL_SMOKE = True after deploy/upload.")

## Full retrieval check

After building or uploading, run the normal experiment path from a terminal or notebook cell:

```bash
uv run python run_experiment.py --backend local --tid lancedb_fts_with_tag_list_devset --batch_size 64
uv run python run_experiment.py --backend modal --tid lancedb_fts_with_tag_list_devset --batch_size 64
```